# Lab 10 – odometry vs gyro-aided localization

The "54" trajectory, driven twice by the same program (`Lab10_odom_gyro.qrs`). Only the course used for localization and control changes:

- run A (`MODE = 0`): `course = course_odom`, from the encoders
- run B (`MODE = 1`): `course = course + gyro_inc`, from the gyro ($b_0$ averaged over 300 samples at rest at the start, $s = 0.35\%$ from Lab 9)

Log format: one line per control cycle, no header, `t,i,theta_l,theta_r,gz,tg,x,y,course`: `t` in ms; `i` = index of the target waypoint; encoder angles in deg; raw `gz` in mdeg/s ($b_0$ not subtracted); `tg` = gyro timestamp (3906 ticks per s); `x`, `y` in cm; `course` in rad (the course the robot used).

- `odom_2.txt` – run A
- `gyro.txt` – run B

In [ ]:
from google.colab import files      # file work module
import numpy as np                  # array work module
import pandas as pd                 # tables
import matplotlib.pyplot as plt     # data visualization

uploaded = files.upload()           # select odom_2.txt and gyro.txt

In [ ]:
cols = ['t_ms', 'i', 'theta_l', 'theta_r', 'gz', 'tg', 'x', 'y', 'course']
runA = pd.read_csv('odom_2.txt', names=cols)    # run A: course from the encoders
runB = pd.read_csv('gyro.txt',   names=cols)    # run B: course from the gyro
for run in (runA, runB):
    run['t'] = (run.t_ms - run.t_ms.iloc[0]) / 1000     # s
    run['seg'] = run.i + 1                              # segment k = rows with target waypoint i = k - 1

for name, run in [('A', runA), ('B', runB)]:
    print(f'run {name}: {len(run)} rows, {run.t.iloc[-1]:.1f} s, one row every {run.t_ms.diff().median():.0f} ms')

## 1. Ideal trajectory

In [ ]:
# Waypoints in cm; point 0 is the start (the robot faces +x)
wp = np.array([[0, 0],
               [50, 0], [50, 50], [0, 50], [0, 100], [50, 100],    # digit 5
               [80, 100],                                          # gap
               [80, 50], [130, 50], [130, 100], [130, 0]])         # digit 4
X_ref, Y_ref = wp[:, 0], wp[:, 1]

def wrap(a):
    return (a + 180) % 360 - 180            # angle to [-180, 180) deg

course_ref = np.degrees(np.arctan2(np.diff(Y_ref), np.diff(X_ref)))    # deg, from +x
path_ref = np.hypot(np.diff(X_ref), np.diff(Y_ref))                    # cm

ref = pd.DataFrame({'seg': np.arange(1, 11),
                    'course_ref': course_ref,
                    'turn': wrap(np.diff(np.r_[0, course_ref])),       # turn before the segment
                    'path_ref': path_ref})
print(ref.to_string(index=False))
print(f'total path_ref = {path_ref.sum():.0f} cm')

10 segments, 530 cm. The hardest turn is the $-180°$ turn before segment 10. There is no turn between segments 5 and 6, which form one 80 cm line.

## 2. Trajectories

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(X_ref, Y_ref, 'k:o', label='ideal')
plt.plot(runA.x, runA.y, label='run A, odometry (logged)')
plt.plot(runB.x, runB.y, label='run B, gyro (logged)')
plt.plot(0, 0, 'kx', markersize=10, mew=2, label='start')
for k in range(10):                                    # segment numbers at the midpoints
    plt.annotate(str(k + 1), ((X_ref[k] + X_ref[k + 1]) / 2, (Y_ref[k] + Y_ref[k + 1]) / 2),
                 textcoords='offset points', xytext=(4, 4), color='gray')
plt.axis('equal'); plt.xlabel('x (cm)'); plt.ylabel('y (cm)'); plt.legend()
plt.show()

*To write:* both logged paths stay close to the ideal "54" (within the distances in §6). Why is that expected, whatever the localization error is? Where do `x` and `y` come from, and what does the controller steer towards? What would show the real error?

## 3. Offline recomputation

In [ ]:
r, B = 0.042, 0.194          # wheel radius, track width (m), from Lab 5
sf = 0.0035                  # gyro scale factor s, from Lab 9
TICKS = 3906                 # gyro timestamp ticks per second (256 us each)

def gyro_factor(run):
    """a_k such that the course increment of row k is a_k * (gz_k - b0), rad"""
    dt = run.tg.diff() / TICKS                          # s
    return dt / 1000 / (1 + sf) * np.pi / 180           # mdeg/s -> rad/s, times dt

# b0 of run B: the logged course is the sum of a_k * (gz_k - b0), so b0 follows from a least-squares fit
a = gyro_factor(runB).iloc[1:]
dc = runB.course.diff().iloc[1:]
b0_B = (a * (a * runB.gz.iloc[1:] - dc)).sum() / (a * a).sum()        # mdeg/s
b0_A = b0_B      # not in run A's log (its course is from the encoders): assumed equal, the runs were ≈ 2 min apart
print(f'b0 (run B) = {b0_B:.1f} mdeg/s = {b0_B / 1000:.3f} deg/s')

In [ ]:
def recompute(run, b0):
    """both localization methods from the raw sensors of one run"""
    rec = pd.DataFrame({'t': run.t, 'seg': run.seg})
    rec['path'] = (run.theta_r + run.theta_l) / 2 * r * np.pi / 180 * 100    # cm
    dpath = rec.path.diff().fillna(rec.path)                                  # cm per row
    rec['c_odom'] = (run.theta_r - run.theta_l) * r / B * np.pi / 180         # rad
    inc = gyro_factor(run) * (run.gz - b0)                                    # rad per row
    inc.iloc[0] = 0                                                           # first row: dt not known
    rec['c_gyro'] = inc.cumsum()                                              # rad
    for m in ('odom', 'gyro'):
        rec['x_' + m] = (dpath * np.cos(rec['c_' + m])).cumsum()              # cm
        rec['y_' + m] = (dpath * np.sin(rec['c_' + m])).cumsum()
    return rec

recA = recompute(runA, b0_A)
recB = recompute(runB, b0_B)

# check: the robot's own estimate is reproduced
for name, run, rec, m in [('A', runA, recA, 'odom'), ('B', runB, recB, 'gyro')]:
    dxy = max((rec['x_' + m] - run.x).abs().max(), (rec['y_' + m] - run.y).abs().max())
    dcourse = np.degrees((rec['c_' + m] - run.course).abs().max())
    print(f'run {name} ({m}): max |x, y difference| {dxy:.3f} cm, max |course difference| {dcourse:.4f} deg')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for a_, rec, name in [(ax[0], recA, 'run A (driven with the encoder course)'),
                      (ax[1], recB, 'run B (driven with the gyro course)')]:
    a_.plot(X_ref, Y_ref, 'k:o', label='ideal')
    a_.plot(rec.x_odom, rec.y_odom, label='course from the encoders')
    a_.plot(rec.x_gyro, rec.y_gyro, label='course from the gyro')
    a_.set_title(name); a_.axis('equal'); a_.set_xlabel('x (cm)'); a_.set_ylabel('y (cm)'); a_.legend()
plt.show()

plt.figure(figsize=(10, 4))
for rec, col, name in [(recA, 'C0', 'run A'), (recB, 'C1', 'run B')]:
    plt.plot(rec.t, np.degrees(rec.c_gyro - rec.c_odom), c=col, label=name)
    for t0 in rec.t[rec.seg.diff() > 0]:               # start of each segment (a turn)
        plt.axvline(t0, c=col, ls=':', lw=0.8)
plt.xlabel('t (s)'); plt.ylabel('gyro − encoder course (deg)'); plt.legend()
plt.show()

for name, rec in [('A', recA), ('B', recB)]:
    diff = np.degrees(rec.c_gyro - rec.c_odom)
    gap = np.hypot(rec.x_gyro.iloc[-1] - rec.x_odom.iloc[-1], rec.y_gyro.iloc[-1] - rec.y_odom.iloc[-1])
    print(f'run {name}: gyro − encoders {diff.iloc[-1]:+.1f} deg at the end (max |.| {diff.abs().max():.1f} deg); '
          f'the two end points are {gap:.1f} cm apart')

Both on-robot estimates are reproduced from the raw sensors (run A exactly; run B to within 0.006 cm and 0.003 deg, since the gyro increment of the first row is not known). $b_0 = 638.2$ mdeg/s ($0.638$ deg/s) in run B; run A's $b_0$ is not in its log, so the gyro course of run A uses run B's value (an assumption).

*To write:* where do the two methods disagree (which segments, which turns), and in which direction? Which error sources from the brief (slip, `B`, bias, scale factor) could explain it?

## 4. Course per segment

In [ ]:
def per_segment(rec, course):
    """course (path-weighted mean, deg) and path increment (cm) of each segment"""
    dpath = rec.path.diff().fillna(rec.path).clip(lower=0)      # cm per row; turning in place moves ≈ 0
    g = pd.DataFrame({'seg': rec.seg, 'w': dpath, 'cw': np.degrees(course) * dpath,
                      'path': rec.path}).groupby('seg')
    course_seg = (g.cw.sum() / g.w.sum()).values
    path_end = g.path.last()
    path_seg = path_end.diff().fillna(path_end).values
    return course_seg, path_seg

cA, pA = per_segment(recA, recA.c_odom)     # run A as driven: encoder course
cB, pB = per_segment(recB, recB.c_gyro)     # run B as driven: gyro course

seg = pd.DataFrame({'seg': ref.seg, 'course_ref': course_ref, 'course_A': cA, 'course_B': cB,
                    'd_course_A': wrap(cA - course_ref), 'd_course_B': wrap(cB - course_ref)})
print(seg.round(1).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 7))
for a_, rec, name in [(ax[0], recA, 'run A'), (ax[1], recB, 'run B')]:
    a_.step(rec.t, course_ref[rec.seg - 1], 'k--', where='post', label='course_ref')
    a_.plot(rec.t, np.degrees(rec.c_odom), label='course_odom')
    a_.plot(rec.t, np.degrees(rec.c_gyro), label='gyro course')
    a_.set_title(name); a_.set_ylabel('course (deg)'); a_.legend()
ax[1].set_xlabel('t (s)')
plt.show()

The course of a segment is the mean weighted by each row's path increment: rows where the robot turns in place hardly change `path`, so they get almost no weight. A plain mean over the segment's rows would mix the turn into the course (up to 19 deg here).

*To write:* the logged course is off `course_ref` by several degrees on most segments, in both runs, although the logged path stays near the ideal. Is that a localization error or the controller's path between the corners (see the curves in §2)?

## 5. Path per segment

In [ ]:
seg['path_ref'] = path_ref
seg['path_A'], seg['path_B'] = pA, pB
seg['d_path_A'], seg['d_path_B'] = pA - path_ref, pB - path_ref
print(seg[['seg', 'path_ref', 'path_A', 'path_B', 'd_path_A', 'd_path_B']].round(1).to_string(index=False))

for name, p in [('A', pA), ('B', pB)]:
    print(f'run {name}: total path {p.sum():.1f} cm vs {path_ref.sum():.0f} cm ({(p.sum() / path_ref.sum() - 1) * 100:+.1f} %)')

*To write:* the path errors of A and B are similar. Why, given how `path` is computed in both runs? Where does the shortfall per segment come from (hint: the 4 cm acceptance radius, and one row every ≈ 130 ms)?

## 6. Error growth

In [ ]:
def cross_track(rec, x, y):
    """distance from each row's position to the ideal segment it is on, cm"""
    k = rec.seg.values - 1
    P0 = np.c_[X_ref[k], Y_ref[k]]                          # segment start
    v = np.c_[X_ref[k + 1], Y_ref[k + 1]] - P0             # segment vector
    w = np.c_[x, y] - P0
    u = np.clip((w * v).sum(1) / (v * v).sum(1), 0, 1)     # nearest point on the segment
    return np.hypot(*(w - u[:, None] * v).T)

err = pd.DataFrame({'seg': ref.seg})
for name, rec in [('A', recA), ('B', recB)]:
    for m in ('odom', 'gyro'):
        ct = pd.Series(cross_track(rec, rec['x_' + m], rec['y_' + m]))
        err[f'{name}_{m}'] = ct.groupby(rec.seg.values).max().values     # max per segment
print(err.round(1).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(seg.seg, seg.d_course_A, 'o-', label='run A')
ax[0].plot(seg.seg, seg.d_course_B, 's-', label='run B')
ax[0].axhline(0, c='gray', lw=0.8); ax[0].set_ylabel('d_course (deg)')
ax[1].plot(err.seg, err.A_odom, 'o-',  c='C0', label='run A, encoders (logged)')
ax[1].plot(err.seg, err.A_gyro, 'o--', c='C0', label='run A, gyro (offline)')
ax[1].plot(err.seg, err.B_gyro, 's-',  c='C1', label='run B, gyro (logged)')
ax[1].plot(err.seg, err.B_odom, 's--', c='C1', label='run B, encoders (offline)')
ax[1].set_ylabel('max distance to the ideal (cm)')
for a_ in ax:
    a_.set_xlabel('segment'); a_.set_xticks(seg.seg); a_.legend()
plt.show()

for name, rec, own, other in [('A', recA, 'odom', 'gyro'), ('B', recB, 'gyro', 'odom')]:
    for m in (own, other):
        x, y = rec['x_' + m].iloc[-1], rec['y_' + m].iloc[-1]
        c = np.degrees(rec['c_' + m].iloc[-1])
        print(f'run {name}, {m} ({"logged" if m == own else "offline"}): end ({x:.1f}, {y:.1f}) cm, '
              f'{np.hypot(x - X_ref[-1], y - Y_ref[-1]):.1f} cm from the last waypoint; '
              f'course {c:.1f} deg, error {wrap(c - course_ref[-1]):+.1f} deg')

The logged estimates end within the 4 cm acceptance radius of the last waypoint by construction: the program stops when its own estimate gets there. Recomputed with the other course, run A ends 19.3 cm and run B 24.3 cm from it.

*To write:* does the error accumulate or stay bounded? After which turn does it jump? Which run's real end point (video) is closer to the last waypoint?

## 7. Comparison table

In [ ]:
table = pd.DataFrame({
    'Seg.': seg.seg.astype(str),
    'course_ref (deg)': course_ref, 'course A (deg)': cA, 'course B (deg)': cB,
    'd_course A (deg)': seg.d_course_A, 'd_course B (deg)': seg.d_course_B,
    'path_ref (cm)': path_ref, 'path A (cm)': pA, 'path B (cm)': pB,
    'd_path A (cm)': pA - path_ref, 'd_path B (cm)': pB - path_ref})
summary = {'Seg.': 'Summary',
           'd_course A (deg)': seg.d_course_A.abs().mean(),     # mean |d_course|
           'd_course B (deg)': seg.d_course_B.abs().mean(),
           'path_ref (cm)': path_ref.sum(), 'path A (cm)': pA.sum(), 'path B (cm)': pB.sum(),
           'd_path A (cm)': pA.sum() - path_ref.sum(), 'd_path B (cm)': pB.sum() - path_ref.sum()}
table = pd.concat([table, pd.DataFrame([summary])], ignore_index=True)
display(table.round(1).fillna(''))

Summary row: mean $|d_{course}|$ over the 10 segments, and the totals (A: $-2.3\%$, B: $-3.5\%$ of 530 cm; §5).

## 8. Analysis

*To write (at most one page):*

1. Which run is closer to the ideal trajectory in course, and which in path? Why are the path errors similar while the course errors are not?
2. Which error source dominated each run: slip, `B`, bias or scale factor? Back it up with the offline recomputation (§3).
3. How would you combine `course_odom` and `gyro_inc`? (Lab 11: the complementary filter.)